# Day 12: Deep Dive into LLM APIs & Building Applications

Building on Day 11, today we'll explore:
1. How the Chat Completions API really works
2. Understanding tokens
3. The "illusion of memory" in LLMs
4. Building a multi-step application with JSON and streaming


## Setup: Import Libraries and Load API Key

Same setup as Day 11 - load our environment and check the API key.


In [1]:
# Import everything we need
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
import tiktoken  # For tokenization

# Load API key from .env file
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if api_key and api_key.startswith("sk-"):
    print("API key loaded successfully!")
else:
    print("Error: Check your .env file")


API key loaded successfully!


In [2]:
# Create OpenAI client - we'll use this throughout
client = OpenAI()

# We'll use gpt-4o-mini for most examples (fast and cheap)
MODEL = "gpt-4o-mini"
print(f"Using model: {MODEL}")


Using model: gpt-4o-mini


## Part 1: Understanding the Chat Completions API

The OpenAI library is just a wrapper around HTTP calls. Let's see what's really happening.


In [3]:
# The standard way to call the API (what we learned in Day 11)
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Tell me a fun fact about space"}
    ]
)

# Get the response
print("Response from GPT:")
print(response.choices[0].message.content)


Response from GPT:
Sure! Did you know that a day on Venus is longer than a year on Venus? It takes about 243 Earth days for Venus to complete a single rotation on its axis (a "day"), while it orbits the Sun in about 225 Earth days (a "year"). This means that a day and a year on Venus are very close in length, but the day is actually longer!


In [4]:
# Let's look at the full response object - it's just structured data!
print("Full response structure:")
print(f"ID: {response.id}")
print(f"Model: {response.model}")
print(f"Choices: {len(response.choices)}")
print(f"Usage - Prompt tokens: {response.usage.prompt_tokens}")
print(f"Usage - Completion tokens: {response.usage.completion_tokens}")
print(f"Usage - Total tokens: {response.usage.total_tokens}")


Full response structure:
ID: chatcmpl-Cx5RomCKIprAkHXaHefHKANHOCqCq
Model: gpt-4o-mini-2024-07-18
Choices: 1
Usage - Prompt tokens: 14
Usage - Completion tokens: 80
Usage - Total tokens: 94


## Part 2: Understanding Tokens

Tokens are the "chunks" of text that LLMs process. Not characters, not words - something in between.


In [5]:
# Get the tokenizer for GPT-4 (works for gpt-4o-mini too)
tokenizer = tiktoken.encoding_for_model("gpt-4")

# Let's tokenize some text
text = "Hello, my name is John and I love AI!"

# Convert text to tokens
tokens = tokenizer.encode(text)

print(f"Original text: {text}")
print(f"Number of tokens: {len(tokens)}")
print(f"Token IDs: {tokens}")


Original text: Hello, my name is John and I love AI!
Number of tokens: 11
Token IDs: [9906, 11, 856, 836, 374, 3842, 323, 358, 3021, 15592, 0]


In [6]:
# Now let's see what each token represents
print("Breaking down each token:")
for i, token_id in enumerate(tokens):
    token_text = tokenizer.decode([token_id])
    print(f"  Token {i}: ID={token_id} -> '{token_text}'")


Breaking down each token:
  Token 0: ID=9906 -> 'Hello'
  Token 1: ID=11 -> ','
  Token 2: ID=856 -> ' my'
  Token 3: ID=836 -> ' name'
  Token 4: ID=374 -> ' is'
  Token 5: ID=3842 -> ' John'
  Token 6: ID=323 -> ' and'
  Token 7: ID=358 -> ' I'
  Token 8: ID=3021 -> ' love'
  Token 9: ID=15592 -> ' AI'
  Token 10: ID=0 -> '!'


In [7]:
# Let's try some interesting examples to see how tokenization works
examples = [
    "Hello",
    "Artificial Intelligence", 
    "3.14159",
    "Supercalifragilisticexpialidocious",
    "OpenAI",
]

print("Token counts for different texts:")
for example in examples:
    tokens = tokenizer.encode(example)
    print(f"  '{example}' -> {len(tokens)} tokens")


Token counts for different texts:
  'Hello' -> 1 tokens
  'Artificial Intelligence' -> 3 tokens
  '3.14159' -> 4 tokens
  'Supercalifragilisticexpialidocious' -> 11 tokens
  'OpenAI' -> 2 tokens


## Part 3: The Illusion of Memory

Here's something important: **Every API call is completely stateless!**

The model doesn't remember anything from previous calls. Let's prove it.


In [8]:
# First call: Tell it our name
response1 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Hi! My name is Alice."}
    ]
)
print("First call - told it my name:")
print(response1.choices[0].message.content)


First call - told it my name:
Hi Alice! It’s great to meet you. How can I assist you today?


In [9]:
# Second call: Ask what our name is (WITHOUT including history)
response2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What's my name?"}
    ]
)
print("Second call - asking for my name (no history):")
print(response2.choices[0].message.content)
print("\n** Notice: It doesn't remember! Each call is independent! **")


Second call - asking for my name (no history):
I'm sorry, but I don't know your name. If you'd like to share it, I'd be happy to address you by it!

** Notice: It doesn't remember! Each call is independent! **


In [10]:
# The trick: Include the conversation history!
# We manually pass all previous messages
response3 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Hi! My name is Alice."},
        {"role": "assistant", "content": "Hello Alice! Nice to meet you. How can I help you today?"},
        {"role": "user", "content": "What's my name?"}
    ]
)
print("Third call - WITH conversation history:")
print(response3.choices[0].message.content)
print("\n** Now it 'remembers' because we passed the history! **")


Third call - WITH conversation history:
Your name is Alice!

** Now it 'remembers' because we passed the history! **


## Part 4: Getting JSON Responses

For structured data, we can force the model to respond in JSON format. This is super useful for applications!


In [11]:
# Ask for JSON output using response_format parameter
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You extract information and respond in JSON format."},
        {"role": "user", "content": """Extract the person's details from this text:
        'John Smith is a 35-year-old software engineer from New York.'
        
        Respond with JSON containing: name, age, occupation, location"""}
    ],
    response_format={"type": "json_object"}  # Force JSON output
)

# The response is a JSON string - let's parse it
json_response = response.choices[0].message.content
print("Raw JSON response:")
print(json_response)

# Parse to Python dict
data = json.loads(json_response)
print("\nParsed as Python dict:")
print(f"  Name: {data.get('name')}")
print(f"  Age: {data.get('age')}")
print(f"  Occupation: {data.get('occupation')}")
print(f"  Location: {data.get('location')}")


Raw JSON response:
{
  "name": "John Smith",
  "age": 35,
  "occupation": "software engineer",
  "location": "New York"
}

Parsed as Python dict:
  Name: John Smith
  Age: 35
  Occupation: software engineer
  Location: New York


## Part 5: Streaming Responses

Streaming gives you the "typewriter effect" - see tokens as they're generated.


In [12]:
# Enable streaming with stream=True
print("Streaming response (watch it appear):\n")

stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Write a short poem about coding (4 lines)"}
    ],
    stream=True  # Enable streaming!
)

# Iterate through chunks as they arrive
full_response = ""
for chunk in stream:
    # Get the content from this chunk (might be None)
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)  # Print without newline
        full_response += content

print("\n\n--- Done streaming! ---")


Streaming response (watch it appear):

In lines of logic, dreams take flight,  
With keys that dance in hopeful light.  
A world of code, where thoughts align,  
In binary whispers, secrets shine.

--- Done streaming! ---


## Part 6: Building a Multi-Step Application

Let's build something practical: A tool that analyzes text and then generates a response based on that analysis.

**The Pattern:**
1. First LLM call: Analyze/Extract (JSON output)
2. Process the result in Python
3. Second LLM call: Generate based on analysis


In [13]:
# Example: Analyze a product review and generate a response

product_review = """
I bought this laptop last month and I'm very disappointed. The battery 
only lasts 2 hours, the screen has dead pixels, and customer support 
was unhelpful. The only good thing is it's lightweight. I want a refund.
"""

print("Original Review:")
print(product_review)
print("\n" + "="*50 + "\n")


Original Review:

I bought this laptop last month and I'm very disappointed. The battery 
only lasts 2 hours, the screen has dead pixels, and customer support 
was unhelpful. The only good thing is it's lightweight. I want a refund.





In [15]:
# STEP 1: Analyze the review (First LLM Call with JSON output)
print("Step 1: Analyzing the review...")

analysis_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You analyze product reviews and extract information in JSON format."},
        {"role": "user", "content": f"""Analyze this product review and respond with JSON:

Review: {product_review}

Include these fields:
- sentiment: "positive", "negative", or "mixed"
- issues: list of problems mentioned
- positives: list of good things mentioned
- action_requested: what the customer wants (e.g., "refund", "replacement", "help")"""}
    ],
    response_format={"type": "json_object"}
)

# Parse the JSON response
analysis = json.loads(analysis_response.choices[0].message.content)

print("Analysis Result:")
print(json.dumps(analysis, indent=2))


Step 1: Analyzing the review...
Analysis Result:
{
  "sentiment": "negative",
  "issues": [
    "battery lasts only 2 hours",
    "screen has dead pixels",
    "unhelpful customer support"
  ],
  "positives": [
    "lightweight"
  ],
  "action_requested": "refund"
}


In [16]:
# STEP 2: Generate a customer service response based on analysis (Second LLM Call)
print("\nStep 2: Generating customer service response...")

# Build context from analysis
issues_text = ", ".join(analysis.get("issues", []))
positives_text = ", ".join(analysis.get("positives", []))
action = analysis.get("action_requested", "assistance")

response_prompt = f"""Based on this analysis of a customer review:
- Sentiment: {analysis.get('sentiment')}
- Issues reported: {issues_text}
- Positive aspects: {positives_text}
- Customer wants: {action}

Write a professional, empathetic customer service response."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful customer service representative. Be empathetic and solution-focused."},
        {"role": "user", "content": response_prompt}
    ]
)

print("\nGenerated Response:")
print("-" * 40)
print(response.choices[0].message.content)



Step 2: Generating customer service response...

Generated Response:
----------------------------------------
Subject: Your Recent Experience with Us

Dear [Customer's Name],

Thank you for taking the time to share your feedback with us. I genuinely regret to hear about your experience with your device, and I appreciate your candor in highlighting the issues you’ve encountered.

I understand how frustrating it must be to deal with a battery that lasts only 2 hours, along with a screen displaying dead pixels. A lightweight device can only bring so much comfort if it doesn’t perform as expected. Moreover, I’m truly sorry to learn about your disappointment with our customer support.

Your satisfaction is important to us, and I want to make this right for you. I would be more than happy to assist you with processing a refund as per your request. Please provide me with your order details, and I will ensure that this matter is resolved as swiftly as possible.

Thank you for your patience an

## Summary

In this notebook, we learned:

1. **The Chat Completions API** is just HTTP calls wrapped nicely
2. **Tokens** are chunks of text - roughly 4 characters or 0.75 words each
3. **Memory is an illusion** - we pass full history with every call
4. **JSON responses** can be forced with `response_format`
5. **Streaming** shows tokens as they generate
6. **Multi-step applications** chain LLM calls for complex tasks

## Your Turn!

Try modifying the code above to:
- Change the product review and see different analyses
- Add a third LLM call (maybe translate the response?)
- Build your own two-step application
